# 財務資訊管理策略二：成交量、KD 低檔買進與多標的回測

本 notebook 使用 `yfinance` 抓取股票日資料，建立一個以 **成交量 + KD 指標 + 分批加碼 + 停損** 為核心的交易策略，並進一步回測多種標的，觀察同一套策略在不同股票與 ETF 上的表現差異。

策略規則：

1. 成交量大於或等於 100,000 股。
2. K 值與 D 值都小於 30 時買進。
3. 已持有時，若價格低於平均買進成本，且 KD 仍小於 30，則加碼。
4. 最多分 3 批進場，避免無限制攤平。
5. K 值與 D 值都大於 70 時全部賣出。
6. 若價格低於平均買進成本 15%，全部停損。
7. 訊號使用前一日資料，於下一個交易日執行，避免未來函數問題。

## 1. 匯入套件

In [ ]:
!pip -q install yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. 參數設定

`MAIN_TICKER` 是單一標的詳細分析使用的股票；`MULTI_TICKERS` 是多標的回測清單。下載成功後會建立快取 CSV，避免一直重複呼叫 Yahoo Finance 而被限流。

In [ ]:
MAIN_TICKER = "TSM"
MULTI_TICKERS = ["TSM", "AAPL", "NVDA", "MSFT", "SPY", "QQQ"]
START_DATE = "2018-01-01"
END_DATE = "2026-05-06"
CACHE_DIR = "price_cache"

VOLUME_THRESHOLD = 100_000
LOW_KD = 30
HIGH_KD = 70
MAX_UNITS = 3
STOP_LOSS = 0.15
TRANSACTION_COST = 0.001425

## 3. 資料下載與預處理

In [ ]:
def download_price_data(ticker, start=START_DATE, end=END_DATE, use_cache=True):
    os.makedirs(CACHE_DIR, exist_ok=True)
    cache_path = os.path.join(CACHE_DIR, f"{ticker}_{start}_{end}.csv")

    if use_cache and os.path.exists(cache_path):
        df = pd.read_csv(cache_path)
        print(f"使用快取資料：{cache_path}")
    else:
        df = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False, threads=False)
        if df.empty:
            raise ValueError(f"無法下載 {ticker}。可能是 Yahoo Finance 暫時限流，請等待幾分鐘後重試。")
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.reset_index()
        df.to_csv(cache_path, index=False)
        print(f"已下載並建立快取：{cache_path}")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").drop_duplicates("Date")
    df = df.set_index("Date")
    price_col = "Adj Close" if "Adj Close" in df.columns else "Close"
    df["Price"] = df[price_col]
    df = df.dropna(subset=["Open", "High", "Low", "Close", "Price", "Volume"])
    df["Return"] = df["Price"].pct_change()
    return df.dropna(subset=["Return"])

raw = download_price_data(MAIN_TICKER)
display(raw.head())
display(raw.tail())
print(f"資料期間：{raw.index.min().date()} 到 {raw.index.max().date()}")
print(f"資料筆數：{len(raw):,}")

## 4. KD 指標計算

KD 指標先計算 RSV：

$$RSV_t = \frac{Close_t - LowestLow_{n,t}}{HighestHigh_{n,t} - LowestLow_{n,t}} \times 100$$

再平滑為 K 與 D：

$$K_t = \frac{2}{3}K_{t-1} + \frac{1}{3}RSV_t$$

$$D_t = \frac{2}{3}D_{t-1} + \frac{1}{3}K_t$$

In [ ]:
def add_kd_indicator(df, window=9):
    out = df.copy()
    low_min = out["Low"].rolling(window).min()
    high_max = out["High"].rolling(window).max()
    out["RSV"] = (out["Close"] - low_min) / (high_max - low_min) * 100
    out["RSV"] = out["RSV"].replace([np.inf, -np.inf], np.nan).fillna(50)

    k_values = []
    d_values = []
    k_prev = 50
    d_prev = 50
    for rsv in out["RSV"]:
        k_now = (2 / 3) * k_prev + (1 / 3) * rsv
        d_now = (2 / 3) * d_prev + (1 / 3) * k_now
        k_values.append(k_now)
        d_values.append(d_now)
        k_prev = k_now
        d_prev = d_now

    out["K"] = k_values
    out["D"] = d_values
    return out

data = add_kd_indicator(raw).dropna()
display(data[["Price", "Volume", "RSV", "K", "D"]].tail())

## 5. 策略回測函數

策略採用分批部位。最多 3 批時，每一批代表三分之一資金；持有 3 批時，代表滿倉。

In [ ]:
def backtest_kd_volume_strategy(df):
    out = df.copy()
    out["Low_KD"] = (out["K"] < LOW_KD) & (out["D"] < LOW_KD)
    out["High_KD"] = (out["K"] > HIGH_KD) & (out["D"] > HIGH_KD)
    out["Volume_Filter"] = out["Volume"] >= VOLUME_THRESHOLD

    units = 0
    avg_cost = np.nan
    units_history = []
    avg_cost_history = []
    action_history = []

    for i in range(len(out)):
        price = out["Price"].iloc[i]
        if i == 0:
            signal_low_kd = False
            signal_high_kd = False
            signal_volume = False
        else:
            signal_low_kd = out["Low_KD"].iloc[i - 1]
            signal_high_kd = out["High_KD"].iloc[i - 1]
            signal_volume = out["Volume_Filter"].iloc[i - 1]

        action = "Hold"
        stop_loss_hit = units > 0 and price <= avg_cost * (1 - STOP_LOSS)

        if units > 0 and (signal_high_kd or stop_loss_hit):
            units = 0
            avg_cost = np.nan
            action = "Sell" if signal_high_kd else "Stop Loss"
        elif units == 0 and signal_volume and signal_low_kd:
            units = 1
            avg_cost = price
            action = "Buy"
        elif units > 0 and units < MAX_UNITS and signal_volume and signal_low_kd and price < avg_cost:
            avg_cost = (avg_cost * units + price) / (units + 1)
            units += 1
            action = "Add"

        units_history.append(units)
        avg_cost_history.append(avg_cost)
        action_history.append(action)

    out["Units"] = units_history
    out["Position"] = out["Units"] / MAX_UNITS
    out["Avg_Cost"] = avg_cost_history
    out["Action"] = action_history
    out["Trade"] = out["Position"].diff().abs().fillna(out["Position"].abs())
    out["Strategy_Return_Gross"] = out["Position"].shift(1).fillna(0) * out["Return"]
    out["Transaction_Cost"] = out["Trade"] * TRANSACTION_COST
    out["Strategy_Return"] = out["Strategy_Return_Gross"] - out["Transaction_Cost"]
    out["BuyHold_Cum"] = (1 + out["Return"]).cumprod()
    out["Strategy_Cum"] = (1 + out["Strategy_Return"]).cumprod()
    return out

result = backtest_kd_volume_strategy(data)
display(result[["Price", "Volume", "K", "D", "Units", "Avg_Cost", "Action", "Strategy_Return"]].tail(15))
display(result["Action"].value_counts().to_frame("次數"))

## 6. 績效衡量函數

In [ ]:
def max_drawdown(cumulative_return):
    running_max = cumulative_return.cummax()
    drawdown = cumulative_return / running_max - 1
    return drawdown.min(), drawdown

def performance_metrics(df, return_col, cumulative_col):
    daily_ret = df[return_col].dropna()
    total_return = df[cumulative_col].iloc[-1] - 1
    years = len(daily_ret) / 252
    annual_return = (1 + total_return) ** (1 / years) - 1
    annual_vol = daily_ret.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol if annual_vol != 0 else np.nan
    mdd, _ = max_drawdown(df[cumulative_col])
    win_rate = (daily_ret > 0).mean()
    return {
        "累積報酬率": total_return,
        "年化報酬率": annual_return,
        "年化波動率": annual_vol,
        "Sharpe Ratio": sharpe,
        "最大回撤": mdd,
        "日勝率": win_rate,
    }

metrics = pd.DataFrame({
    "Buy and Hold": performance_metrics(result, "Return", "BuyHold_Cum"),
    "KD Volume Strategy": performance_metrics(result, "Strategy_Return", "Strategy_Cum"),
}).T

display(metrics.style.format("{:.2%}", subset=["累積報酬率", "年化報酬率", "年化波動率", "最大回撤", "日勝率"]).format("{:.2f}", subset=["Sharpe Ratio"]))

## 7. 單一標的圖表

In [ ]:
def plot_kd_strategy(df, ticker):
    fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True, gridspec_kw={"height_ratios": [3, 1.6, 1.1, 2]})
    buys = df[df["Action"] == "Buy"]
    adds = df[df["Action"] == "Add"]
    sells = df[df["Action"] == "Sell"]
    stops = df[df["Action"] == "Stop Loss"]

    axes[0].plot(df.index, df["Price"], label="Adjusted Price", color="#1f77b4")
    axes[0].plot(df.index, df["Avg_Cost"], label="Average Cost", color="#ff7f0e", linestyle="--")
    axes[0].scatter(buys.index, buys["Price"], marker="^", color="green", s=70, label="Buy")
    axes[0].scatter(adds.index, adds["Price"], marker="^", color="#2ca02c", s=45, label="Add")
    axes[0].scatter(sells.index, sells["Price"], marker="v", color="red", s=65, label="Sell")
    axes[0].scatter(stops.index, stops["Price"], marker="x", color="black", s=70, label="Stop Loss")
    axes[0].set_title(f"{ticker} KD Volume Averaging Strategy")
    axes[0].set_ylabel("Price")
    axes[0].legend(loc="upper left")

    axes[1].plot(df.index, df["K"], label="K", color="#9467bd")
    axes[1].plot(df.index, df["D"], label="D", color="#8c564b")
    axes[1].axhline(70, color="red", linestyle="--", linewidth=0.9, label="High KD 70")
    axes[1].axhline(30, color="green", linestyle="--", linewidth=0.9, label="Low KD 30")
    axes[1].set_ylim(0, 100)
    axes[1].set_title("KD Indicator")
    axes[1].set_ylabel("KD")
    axes[1].legend(loc="upper left")

    axes[2].step(df.index, df["Position"], where="post", label="Position", color="#17becf")
    axes[2].set_ylim(-0.05, 1.05)
    axes[2].set_title("Position Size")
    axes[2].set_ylabel("Exposure")

    axes[3].plot(df.index, df["BuyHold_Cum"], label="Buy and Hold", color="#7f7f7f")
    axes[3].plot(df.index, df["Strategy_Cum"], label="KD Volume Strategy", color="#17becf")
    axes[3].set_title("Cumulative Return Comparison")
    axes[3].set_ylabel("Growth of $1")
    axes[3].legend(loc="upper left")

    plt.xlabel("Date")
    plt.tight_layout()
    plt.show()

plot_kd_strategy(result, MAIN_TICKER)

## 8. 多標的回測

為了檢查策略是否只對單一股票有效，本研究另外選擇 TSM、AAPL、NVDA、MSFT、SPY 與 QQQ 進行多標的回測。這些標的包含個股與 ETF，可以觀察同一套 KD 低檔買進、跌破成本加碼、KD 高檔賣出與 15% 停損策略，在不同資產上的表現差異。

In [ ]:
multi_results = {}
summary_rows = []
failed_tickers = []

for ticker in MULTI_TICKERS:
    try:
        print(f"回測中：{ticker}")
        df = download_price_data(ticker)
        df = add_kd_indicator(df).dropna()
        bt = backtest_kd_volume_strategy(df)
        multi_results[ticker] = bt

        row = performance_metrics(bt, "Strategy_Return", "Strategy_Cum")
        row["Ticker"] = ticker
        row["買進次數"] = int((bt["Action"] == "Buy").sum())
        row["加碼次數"] = int((bt["Action"] == "Add").sum())
        row["KD高檔賣出次數"] = int((bt["Action"] == "Sell").sum())
        row["停損次數"] = int((bt["Action"] == "Stop Loss").sum())
        row["總交易動作次數"] = int((bt["Action"] != "Hold").sum())
        row["買進持有累積報酬率"] = bt["BuyHold_Cum"].iloc[-1] - 1
        row["策略是否勝出買進持有"] = bt["Strategy_Cum"].iloc[-1] > bt["BuyHold_Cum"].iloc[-1]
        summary_rows.append(row)
    except Exception as err:
        failed_tickers.append((ticker, str(err)))
        print(f"{ticker} 回測失敗：{err}")

multi_summary = pd.DataFrame(summary_rows).set_index("Ticker")
display(
    multi_summary.style
    .format("{:.2%}", subset=["累積報酬率", "年化報酬率", "年化波動率", "最大回撤", "日勝率", "買進持有累積報酬率"])
    .format("{:.2f}", subset=["Sharpe Ratio"])
)

if failed_tickers:
    display(pd.DataFrame(failed_tickers, columns=["Ticker", "失敗原因"]))

## 9. 多標的視覺化比較

In [ ]:
plt.figure(figsize=(15, 6))
for ticker, bt in multi_results.items():
    plt.plot(bt.index, bt["Strategy_Cum"], label=f"{ticker} Strategy")
plt.title("KD Volume Averaging Strategy: Multi-Asset Backtest")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.show()

return_compare = multi_summary[["累積報酬率", "買進持有累積報酬率"]].copy()
return_compare = return_compare.rename(columns={"累積報酬率": "策略累積報酬率"})
return_compare.plot(kind="bar", figsize=(12, 5), title="Strategy vs Buy and Hold Total Return")
plt.ylabel("Total Return")
plt.xticks(rotation=0)
plt.show()

## 10. 交易紀錄整理

In [ ]:
trade_log = result[result["Action"] != "Hold"][["Price", "Volume", "K", "D", "Units", "Avg_Cost", "Action"]].copy()
display(trade_log.tail(30))

print(f"買進次數：{(trade_log['Action'] == 'Buy').sum()}")
print(f"加碼次數：{(trade_log['Action'] == 'Add').sum()}")
print(f"KD 高檔賣出次數：{(trade_log['Action'] == 'Sell').sum()}")
print(f"停損次數：{(trade_log['Action'] == 'Stop Loss').sum()}")

## 11. 策略結果說明

這個策略屬於反向操作策略：當 KD 低於 30 時，代表價格相對近期高低點偏弱，因此嘗試在低檔區買進。成交量大於 100,000 的條件則用來排除流動性太差的交易日，避免買賣訊號出現在成交稀少、價格容易失真的情況。

加碼邏輯是本策略的重點。若價格低於平均買進成本，但 KD 仍在低檔，代表策略認為價格仍可能處於超跌區，因此以分批方式降低平均成本。不過加碼也會放大風險，所以本 notebook 限制最多 3 批，並加入 15% 停損，避免股價持續下跌時無限制攤平。

多標的回測可以檢查策略是否具有穩定性。若策略在多數標的上都能降低最大回撤或提高 Sharpe Ratio，代表此策略可能有較好的風險控制效果。若只在少數標的表現良好，則表示策略可能較依賴個別股票特性，不一定能直接套用到所有市場。

## 12. 結論

本策略結合成交量過濾、KD 低檔買進、低於成本加碼、KD 高檔出場與 15% 停損。相較於單純買進持有，這個策略更重視進出場時機與風險控制。

不過，KD 是震盪型指標，在明顯趨勢市場中可能產生錯誤訊號。未來可以加入長期均線濾網，例如只在價格高於 200 日均線時買進，或測試不同 KD 門檻、加碼次數與停損比例，觀察策略是否更穩定。